# Amélioration de l’approche de classification

## Prérequis

Cette
étape
repose
sur
la
finalisation
de
l’étape
3.

Les
données
préparées
pour
la
modélisation
sont
disponibles, ainsi
qu’un
premier
benchmark
de
classification
comprenant
un
modèle
Dummy, un
modèle
linéaire
et
un
modèle
non - linéaire.

## Objectif

L’objectif
de
cette
étape
est
d’améliorer
la
modélisation
en
tenant
compte:

- du
déséquilibre
de
la
variable
cible;
- du
contexte
métier;
- de
la
nécessité
d’évaluer
le
modèle
de
manière
plus
robuste.

## Approche retenue

La
démarche
adoptée
dans
ce
notebook
est
la
suivante:

1.
reconstruire
les
données
de
modélisation;
2.
enrichir
les
features
par
un
peu
de
feature
engineering;
3.
réaliser
une
séparation
stratifiée
en
train
et
test;
4.
entraîner
un
modèle
non - linéaire
pondéré;
5.
évaluer
le
modèle
en
validation
croisée
stratifiée;
6.
utiliser
la
courbe
précision - rappel
pour
choisir
un
seuil
de
décision
plus
adapté
au
besoin
métier.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    average_precision_score,
)

pd.set_option ("display.max_columns", 200)
sns.set_theme (style="whitegrid")

## 1. Import des bibliothèques

Cette étape charge les outils nécessaires au feature engineering, à la stratification, à l’entraînement du modèle non-linéaire et à l’évaluation avancée.

In [ ]:
PROJECT_ROOT = Path.cwd ( ).resolve ( ).parent if Path.cwd ( ).name == "notebooks" else Path.cwd ( ).resolve ( )
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

df_central = pd.read_csv (DATA_PROCESSED / "df_central.csv")

df_central["a_quitte_l_entreprise"] = (
    df_central["a_quitte_l_entreprise"]
    .astype (str)
    .str.strip ( )
    .str.capitalize ( )
)

df_central["attrition_bin"] = df_central["a_quitte_l_entreprise"].apply (
    lambda x: 1 if x == "Oui" else 0
)

print (df_central.shape)
display (df_central.head ( ))

In [ ]:
def split_features_by_type(df: pd.DataFrame):
    num_cols = df.select_dtypes (include=["number"]).columns.tolist ( )
    cat_cols = df.select_dtypes (include=["object", "string", "category"]).columns.tolist ( )
    return num_cols, cat_cols


def build_target(df: pd.DataFrame, target_col: str = "attrition_bin") -> pd.Series:
    return df[target_col].copy ( )


def build_feature_matrix(
        df: pd.DataFrame,
        cols_to_drop: list[str] | None = None
) -> pd.DataFrame:
    if cols_to_drop is None:
        cols_to_drop = [
            "a_quitte_l_entreprise",
            "attrition_bin",
            "id_employee",
            "code_sondage",
            "eval_number",
        ]
    return df.drop (columns=[col for col in cols_to_drop if col in df.columns]).copy ( )


def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy ( )
    num_cols, cat_cols = split_features_by_type (df)

    if not cat_cols:
        return df

    encoder = OneHotEncoder (
        drop="first",
        handle_unknown="ignore",
        sparse_output=False
    )

    encoded_array = encoder.fit_transform (df[cat_cols])

    encoded_df = pd.DataFrame (
        encoded_array,
        columns=encoder.get_feature_names_out (cat_cols),
        index=df.index
    )

    return pd.concat ([df[num_cols], encoded_df], axis=1)


def drop_highly_correlated_features(df: pd.DataFrame, threshold: float = 0.8):
    corr_matrix = df.corr (numeric_only=True).abs ( )
    upper = corr_matrix.where (np.triu (np.ones (corr_matrix.shape), k=1).astype (bool))
    to_drop = [column for column in upper.columns if any (upper[column] > threshold)]
    return df.drop (columns=to_drop), to_drop

In [ ]:
df_fe = df_central.copy ( )

df_fe["revenu_par_annee_experience"] = (
        df_fe["revenu_mensuel"] / (df_fe["annee_experience_totale"] + 1)
)

df_fe["part_anciennete_entreprise"] = (
        df_fe["annees_dans_l_entreprise"] / (df_fe["annee_experience_totale"] + 1)
)

df_fe["part_anciennete_poste"] = (
        df_fe["annees_dans_le_poste_actuel"] / (df_fe["annees_dans_l_entreprise"] + 1)
)

df_fe["charge_distance"] = (
        df_fe["distance_domicile_travail"] * df_fe["nombre_heures_travailless"]
)

satisfaction_cols = [
    "satisfaction_employee_environnement",
    "satisfaction_employee_nature_travail",
    "satisfaction_employee_equipe",
    "satisfaction_employee_equilibre_pro_perso",
]

df_fe["score_satisfaction_moyen"] = df_fe[satisfaction_cols].mean (axis=1)

## 2. Feature engineering

Quelques variables dérivées sont ajoutées afin d’enrichir la représentation initiale des employés.

L’objectif n’est pas de multiplier artificiellement les features, mais de créer des variables synthétiques plausibles au regard du contexte métier, par exemple en combinant revenu, expérience, ancienneté, charge de travail ou satisfaction.

In [ ]:
y = build_target (df_fe)
X_raw = build_feature_matrix (df_fe)
X_prepared = encode_features (X_raw)
X, dropped_corr_features = drop_highly_correlated_features (X_prepared, threshold=0.8)

print ("Shape final de X :", X.shape)
print ("Shape final de y :", y.shape)
print ("NaN dans X :", X.isna ( ).sum ( ).sum ( ))
print ("NaN dans y :", y.isna ( ).sum ( ))
print ("Variables supprimées :", dropped_corr_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split (
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print ("X_train :", X_train.shape)
print ("X_test  :", X_test.shape)
print ("y_train :", y_train.shape)
print ("y_test  :", y_test.shape)

print ("\nRépartition y_train :")
print (y_train.value_counts (normalize=True))

print ("\nRépartition y_test :")
print (y_test.value_counts (normalize=True))

## 3. Séparation stratifiée des données

Les jeux d’apprentissage et de test sont construits à l’aide d’une stratification sur la cible afin de conserver une répartition comparable des classes dans les deux sous-ensembles.

In [ ]:
rf_model = RandomForestClassifier (
    n_estimators=400,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

## 4. Modèle non-linéaire retenu

Le modèle retenu est une Random Forest pondérée par `class_weight="balanced"` afin de mieux tenir compte du déséquilibre de la variable cible.

L’objectif est d’améliorer la détection des départs tout en limitant le surapprentissage observé à l’étape précédente.

In [ ]:
def evaluate_cv_model(model, X, y, n_splits=5):
    skf = StratifiedKFold (n_splits=n_splits, shuffle=True, random_state=42)

    fold_metrics = []

    for fold_idx, (train_idx, valid_idx) in enumerate (skf.split (X, y), start=1):
        X_fold_train = X.iloc[train_idx]
        X_fold_valid = X.iloc[valid_idx]
        y_fold_train = y.iloc[train_idx]
        y_fold_valid = y.iloc[valid_idx]

        model.fit (X_fold_train, y_fold_train)

        y_pred_train = model.predict (X_fold_train)
        y_pred_valid = model.predict (X_fold_valid)

        fold_metrics.append ({
            "fold": fold_idx,
            "train_accuracy": accuracy_score (y_fold_train, y_pred_train),
            "train_precision": precision_score (y_fold_train, y_pred_train, zero_division=0),
            "train_recall": recall_score (y_fold_train, y_pred_train, zero_division=0),
            "train_f1": f1_score (y_fold_train, y_pred_train, zero_division=0),
            "valid_accuracy": accuracy_score (y_fold_valid, y_pred_valid),
            "valid_precision": precision_score (y_fold_valid, y_pred_valid, zero_division=0),
            "valid_recall": recall_score (y_fold_valid, y_pred_valid, zero_division=0),
            "valid_f1": f1_score (y_fold_valid, y_pred_valid, zero_division=0),
        })

    return pd.DataFrame (fold_metrics)

In [ ]:
cv_results = evaluate_cv_model (rf_model, X_train, y_train, n_splits=5)
display (cv_results)

In [ ]:
cv_summary = cv_results.drop (columns="fold").agg (["mean", "std"]).T
display (cv_summary)

## 5. Validation croisée stratifiée

Une validation croisée stratifiée est utilisée afin d’obtenir une estimation plus robuste de la performance du modèle.

Les métriques sont stockées pour chaque fold, puis synthétisées à l’aide de la moyenne et de l’écart-type. Cela permet d’évaluer :

- la stabilité du modèle ;
- sa capacité de généralisation ;
- l’existence éventuelle d’un surapprentissage.

In [ ]:
rf_model.fit (X_train, y_train)

y_pred_train = rf_model.predict (X_train)
y_pred_test = rf_model.predict (X_test)

results_final = pd.DataFrame ([
    {
        "split": "train",
        "accuracy": accuracy_score (y_train, y_pred_train),
        "precision": precision_score (y_train, y_pred_train, zero_division=0),
        "recall": recall_score (y_train, y_pred_train, zero_division=0),
        "f1": f1_score (y_train, y_pred_train, zero_division=0),
    },
    {
        "split": "test",
        "accuracy": accuracy_score (y_test, y_pred_test),
        "precision": precision_score (y_test, y_pred_test, zero_division=0),
        "recall": recall_score (y_test, y_pred_test, zero_division=0),
        "f1": f1_score (y_test, y_pred_test, zero_division=0),
    },
]).round (3)

display (results_final)

In [ ]:
print ("TRAIN")
print (classification_report (y_train, y_pred_train, zero_division=0))

print ("TEST")
print (classification_report (y_test, y_pred_test, zero_division=0))

In [ ]:
fig, axes = plt.subplots (1, 2, figsize=(10, 4))

ConfusionMatrixDisplay.from_predictions (y_train, y_pred_train, ax=axes[0], cmap="Blues")
axes[0].set_title ("RandomForest améliorée - Train")

ConfusionMatrixDisplay.from_predictions (y_test, y_pred_test, ax=axes[1], cmap="Blues")
axes[1].set_title ("RandomForest améliorée - Test")

plt.tight_layout ( )
plt.show ( )

In [ ]:
y_scores_test = rf_model.predict_proba (X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve (y_test, y_scores_test)
ap_score = average_precision_score (y_test, y_scores_test)

plt.figure (figsize=(8, 5))
plt.plot (recall, precision, label=f"AP = {ap_score:.3f}")
plt.xlabel ("Recall")
plt.ylabel ("Precision")
plt.title ("Courbe précision-rappel")
plt.legend ( )
plt.show ( )

In [ ]:
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)

threshold_df = pd.DataFrame ({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1],
    "f1": f1_scores,
})

best_threshold_row = threshold_df.sort_values ("f1", ascending=False).iloc[0]
display (best_threshold_row)

In [ ]:
best_threshold = best_threshold_row["threshold"]

y_pred_test_threshold = (y_scores_test >= best_threshold).astype (int)

threshold_results = pd.DataFrame ([{
    "accuracy": accuracy_score (y_test, y_pred_test_threshold),
    "precision": precision_score (y_test, y_pred_test_threshold, zero_division=0),
    "recall": recall_score (y_test, y_pred_test_threshold, zero_division=0),
    "f1": f1_score (y_test, y_pred_test_threshold, zero_division=0),
}]).round (3)

display (threshold_results)

In [ ]:
ConfusionMatrixDisplay.from_predictions (y_test, y_pred_test_threshold, cmap="Blues")
plt.title (f"RandomForest - Test avec seuil optimisé ({best_threshold:.3f})")
plt.show ( )

## 6. Ajustement du seuil de décision

La courbe précision-rappel permet d’ajuster le seuil de classification au lieu de conserver automatiquement le seuil par défaut de 0.5.

Cette étape est importante dans un contexte déséquilibré, car elle permet d’adapter le compromis entre precision et recall au besoin métier.

Si l’objectif prioritaire est d’identifier le plus grand nombre possible d’employés à risque de départ, un seuil favorisant le recall peut être pertinent, au prix d’une baisse de précision.

## 7. Interprétation des performances du modèle amélioré

La validation croisée montre que le modèle non-linéaire amélioré présente encore un écart entre les performances d’apprentissage et de validation, ce qui suggère un surapprentissage modéré. Néanmoins, cet écart reste nettement plus maîtrisé que lors de l’étape précédente.

Avec le seuil de décision standard, le modèle obtient une accuracy correcte sur le jeu de test, mais son recall sur la classe positive reste relativement faible. Cela signifie qu’une part importante des employés quittant réellement l’entreprise n’est pas détectée.

L’utilisation de la courbe précision-rappel pour ajuster le seuil de décision permet d’obtenir un compromis plus cohérent avec le contexte métier. Après optimisation du seuil, le modèle atteint un meilleur équilibre entre precision et recall, avec un score F1 sensiblement amélioré.

Ce résultat montre l’intérêt d’adapter le seuil de classification dans un contexte déséquilibré, plutôt que de conserver automatiquement le seuil par défaut de 0.5.

## Conclusion

Cette étape a permis d’améliorer l’approche de classification en tenant compte du déséquilibre des classes et du besoin métier.

Les principales améliorations apportées sont :

- une séparation stratifiée des données ;
- l’ajout de features dérivées ;
- l’utilisation d’un modèle non-linéaire pondéré ;
- une validation croisée stratifiée ;
- l’ajustement du seuil de décision à l’aide de la courbe précision-rappel.

Le modèle obtenu reste imparfait, mais il offre un compromis plus pertinent que les approches précédentes, en particulier après optimisation du seuil. Il constitue donc une base plus robuste pour la suite du projet et pour l’interprétation des facteurs de départ.